# 19. 여름 사진은 왜 달라졌나 (실험 7 후속 ②, 학습 없음)

**한 줄 요약:** 18번 결과, 가짜 변화의 주범은 **여름 사진**이었다(겨울·눈 아님). 2022–23 여름 사진은 숲이 덜 푸르게(근적외선·NDVI 낮게) 찍혀 모델이 침엽을 더 많이 찍었다. 이 노트북은 **"사진 만드는 방식을 바꾸면 그 차이가 줄어드나"**를 GEE에서 바로 잰다. 재학습 없음.

**용의자 셋 — 하나씩 바꿔서 두 시기 여름 숲 NDVI 차이가 줄어드는지 본다**
- **① 덜 걸러진 옅은 구름·연무** — 지금 구름 마스크(SCL)는 옅은 구름을 놓친다. 2022–23 여름은 파랑(B2) 밴드가 밝고 근적외선이 어두움 = 연무 모양 [추정]. → 더 정교한 구름 마스크(Cloud Score+)를 씌운다
- **② 새 위성 Sentinel-2C** — 2024–25 여름 장면의 9–19%가 S2C, 2022–23은 0% (18 [6]). → 2024–25에서 S2C를 뺀다
- **③ 어느 달 사진이 많이 들어갔나** — 6–9월 중 맑은 날이 많은 달이 시기마다 다르면 중앙값이 달라진다. → 달마다 같은 무게로 합성한다

**"모두 실행" 가능.** [2]에서 `CONFIRMED = True`로 바꾸기 전에는 멈춘다. [4]에서 GEE 로그인 창이 뜰 수 있다. 결과는 지역·시기마다 저장되므로 끊기거나 시간 초과가 나면 [4]만 다시 실행하면 된다. 소요 20–60분 [추정].

In [3]:
# [1] 설정
!pip install -q rasterio earthengine-api

import os, json, time
import numpy as np
import pandas as pd
import rasterio
import ee
from google.colab import drive

drive.mount('/content/drive')
D       = '/content/drive/MyDrive/forest/data'
OUT     = '/content/drive/MyDrive/forest/outputs'
PR_PATH = f'{OUT}/exp19_prereg.json'
RES     = f'{OUT}/exp19_gee.json'
PROJECT_ID = 'driven-era-467023-m3'

TRAIN_REGIONS = ['inje', 'hongcheon', 'gangneung', 'pyeongchang',
                 'bonghwa', 'geumsan', 'geochang', 'suncheon']
KO = {'inje': '인제', 'hongcheon': '홍천', 'gangneung': '강릉', 'pyeongchang': '평창',
      'bonghwa': '봉화', 'geumsan': '금산', 'geochang': '거창', 'suncheon': '순천'}
BIG = ['inje', 'hongcheon', 'geumsan', 'geochang', 'suncheon']     # 17에서 가짜 변화 ≥ +1.0 (18과 같은 5곳)

PERIODS = {'new': ('2024-01-01', '2025-12-31'), 'old': ('2022-01-01', '2023-12-31')}
SUMMER  = [6, 7, 8, 9]                       # v1과 같음
CSP_TH  = 0.60                               # Cloud Score+ 기준 (GEE 안내 예시값)
SCALE   = 60

for r in TRAIN_REGIONS:
    assert os.path.exists(f'{D}/s2/{r}_s2_20band.tif'), f'v1 영상 없음: {r}'
assert os.path.exists(f'{OUT}/exp18_A_z_by_feature.csv'), '18번 결과 없음'
print('파일 확인 완료')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
파일 확인 완료


In [4]:
# [2] 사전 기록 — 결과를 보기 전에 확정. CONFIRMED = True 로 바꿔야 진행된다
CONFIRMED = True

PREREG = {
    'date': time.strftime('%Y-%m-%d %H:%M'),
    'state_before': ('18번 결과(A·B·C)를 봄: 여름 주원인(E_S 0.61–0.65), 겨울은 반대 방향으로 일부 상쇄, 눈 차이 없음. '
                     '2022–23 여름은 숲 근적외선(B6–B8A) z −0.2~−0.6, s_ndvi 6곳 낮음(강릉·평창만 높음), 남부 파랑(B2) 높음. '
                     '이 노트북의 GEE 값은 보지 않음'),
    'question': '두 시기 여름 숲 NDVI 차이가 구름 마스크 / S2C 제외 / 달 균등 합성 중 무엇으로 줄어드는가',

    'measure': ('지역·시기별 여름(6–9월) 합성의 숲 평균값. 숲 = Hansen treecover2000 ≥ 50 & 손실 기록 없음(18 [6]과 같음). '
                f'scale {SCALE} m. 차이 gap = 2022–23 − 2024–25 (NDVI). '
                'base = v1 방식(SCL 3·8·9·10 제외, 구름 50% 미만, 밴드 중앙값 후 NDVI). '
                f'csp = base + Cloud Score+ cs_cdf ≥ {CSP_TH} (두 시기 모두). '
                'noS2C = 2024–25에서 Sentinel-2C 제외(2022–23은 base 그대로). '
                'month = 6·7·8·9월 각각 중앙값 → 네 달 평균 (두 시기 모두)'),
    'check': ('base gap 부호가 18 캐시의 s_ndvi 이동(z) 부호와 8곳 중 7곳 이상 같아야 이 측정을 믿는다. '
              '아니면 판정하지 않고 보고'),
    'rule': ('용의자 X 지지 = 가짜 변화 큰 5곳(인제·홍천·금산·거창·순천) 중 4곳 이상에서 '
             '|gap_X| ≤ 0.5 × |gap_base|. 여러 개 지지면 5곳 평균 줄어든 비율이 큰 순서. '
             '0.5 근거: 18에서 여름 몫 설명률 기준 0.6과 같은 "분명한 다수" 수준, 반 이상 사라져야 주원인으로 봄'),
    'record_only': ['B8(근적외선)·B2(파랑)·B11 gap, 변형별', 'AOT(대기 에어로졸) 합성 평균',
                    '유효 관측 중 SCL 7(구름 가능성 낮음/미분류) 비율 — 옅은 구름이 남는 칸',
                    '달별 유효 관측 수(6·7·8·9월)', 'csp 적용 시 유효 관측이 남는 비율'],
    'predictions_추정': ['① csp 지지 가능성이 가장 큼 (파랑↑·근적외선↓ = 연무 모양)',
                         '② noS2C 지지 안 됨 (S2C 비중 9–19%라 중앙값을 크게 못 움직임)',
                         '③ month는 모름'],
    'next_step': {
        'csp': '구름 마스크를 Cloud Score+로 바꿔 두 시기 다시 받기 → 재학습 → 실험 7 다시 (무거움: v1 8곳 + 2022–23 8곳 다시 내보내기)',
        'noS2C': 'R3: S2C 뺀 2024–25 다시 받기 → 재학습 → 실험 7 다시',
        'month': '달마다 같은 무게로 합성해 두 시기 다시 받기 → 재학습 → 실험 7 다시',
        '없음': ('사진 만드는 방식으로는 못 줄임 = 그해 실제 숲·날씨 차이 쪽. '
                 '20 = 두 시기 사진을 섞어 학습(데이터 추가 없음) → 가짜 변화가 줄어드는지'),
    },
}

if os.path.exists(PR_PATH):
    print('저장된 사전 기록 (덮어쓰지 않음):\n')
    print(json.dumps(json.load(open(PR_PATH, encoding='utf-8')), ensure_ascii=False, indent=1))
else:
    assert CONFIRMED, '사전 기록을 읽고 CONFIRMED = True 로 바꾼 뒤 다시 실행하세요 (정상 정지)'
    json.dump(PREREG, open(PR_PATH, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    print('사전 기록 저장:', PR_PATH)

사전 기록 저장: /content/drive/MyDrive/forest/outputs/exp19_prereg.json


In [5]:
# [3] GEE 함수
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate(); ee.Initialize(project=PROJECT_ID)

HANSEN_IDS = ['UMD/hansen/global_forest_change_2024_v1_12', 'UMD/hansen/global_forest_change_2025_v1_13',
              'UMD/hansen/global_forest_change_2023_v1_11']
def hansen():
    for hid in HANSEN_IDS:
        try:
            img = ee.Image(hid); img.bandNames().getInfo(); return img
        except Exception:
            continue
    raise RuntimeError('Hansen 자료를 찾지 못함 — 멈추고 보고')
HZ = hansen()
FOREST = HZ.select('treecover2000').gte(50).And(HZ.select('lossyear').unmask(0).eq(0))
CSP = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
BANDS = ['B2', 'B4', 'B8', 'B11']

def aoi_of(name):
    with rasterio.open(f'{D}/s2/{name}_s2_20band.tif') as s:
        crs, b = f'EPSG:{s.crs.to_epsg()}', s.bounds
    return ee.Geometry.Rectangle([b.left, b.bottom, b.right, b.top], proj=crs, geodesic=False), crs

def raw_col(aoi, period, months, drop_s2c=False, csp=False):
    mf = ee.Filter.Or([ee.Filter.calendarRange(m, m, 'month') for m in months])
    c = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(aoi)
         .filterDate(*period).filter(mf).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50)))
    if drop_s2c:
        c = c.filter(ee.Filter.neq('SPACECRAFT_NAME', 'Sentinel-2C'))
    if csp:
        c = c.linkCollection(CSP, ['cs_cdf'])
    return c

def scl_good(img):
    scl = img.select('SCL')
    return scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))     # v1 cloud_mask와 같음

def masked(c, csp=False):
    def f(img):
        good = scl_good(img)
        if csp:
            good = good.And(img.select('cs_cdf').gte(CSP_TH))
        return (img.select(BANDS).divide(10000)
                .addBands(img.select('AOT').multiply(0.001).rename('AOT'))
                .updateMask(good))
    return c.map(f)

def with_ndvi(comp, tag):
    nd = comp.normalizedDifference(['B8', 'B4']).rename('NDVI')    # v1처럼 합성한 밴드로 NDVI
    return comp.select(BANDS + ['AOT']).addBands(nd).rename([f'{tag}_{b}' for b in BANDS + ['AOT', 'NDVI']])

def summer_stats(name, pn):
    aoi, crs = aoi_of(name)
    per = PERIODS[pn]
    imgs = [with_ndvi(masked(raw_col(aoi, per, SUMMER)).median(), 'base'),
            with_ndvi(masked(raw_col(aoi, per, SUMMER, csp=True), csp=True).median(), 'csp')]
    monthly = [masked(raw_col(aoi, per, [m])).median() for m in SUMMER]
    imgs.append(with_ndvi(ee.ImageCollection(monthly).mean(), 'month'))
    if pn == 'new':
        imgs.append(with_ndvi(masked(raw_col(aoi, per, SUMMER, drop_s2c=True)).median(), 'noS2C'))

    # 관측 수 기록: 달별 유효 관측, SCL 7 비율, csp 후 남는 비율
    def cnt(img):
        g = scl_good(img)
        return g.rename('v').addBands(g.And(img.select('SCL').eq(7)).rename('s7')).toFloat()
    ss = raw_col(aoi, per, SUMMER).map(cnt).sum()
    rec = [ss.select('s7').divide(ss.select('v').max(1)).rename('rec_scl7share')]
    for m in SUMMER:
        rec.append(raw_col(aoi, per, [m]).map(lambda i: scl_good(i).rename('v').toFloat()).sum().rename(f'rec_nvalid_m{m}'))
    cc = raw_col(aoi, per, SUMMER, csp=True).map(
        lambda i: scl_good(i).And(i.select('cs_cdf').gte(CSP_TH)).rename('v').toFloat()).sum()
    rec.append(cc.divide(ss.select('v').max(1)).rename('rec_csp_keep'))

    img = ee.Image.cat(imgs + rec).updateMask(FOREST)
    red = img.reduceRegion(ee.Reducer.mean(), aoi, scale=SCALE, crs=crs, maxPixels=1e10,
                           bestEffort=True, tileScale=8)
    n_img = raw_col(aoi, per, SUMMER).size()
    return ee.Dictionary(dict(stats=red, n_img=n_img)).getInfo()
print('함수 정의 완료')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python
/usr/local/lib/python3.13/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for UMD/hansen/global_forest_change_2024_v1_12! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by UMD/hansen/global_forest_change_2025_v1_13

Learn more: https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2024_v1_12

  warnings.warn(warning, category=DeprecationWarning)


함수 정의 완료


In [6]:
# [4] 지역·시기별 계산 — 하나 끝날 때마다 저장. 실패하면 이 셀만 다시 실행 (남은 것만 진행)
assert os.path.exists(PR_PATH), '사전 기록 [2]를 먼저 확정하세요 (정상 정지)'
res = json.load(open(RES)) if os.path.exists(RES) else {}
for r in TRAIN_REGIONS:
    for pn in ('new', 'old'):
        key = f'{r}|{pn}'
        if key in res: continue
        t0 = time.time()
        try:
            res[key] = summer_stats(r, pn)
            json.dump(res, open(RES, 'w'), ensure_ascii=False, indent=1)
            s = res[key]['stats']
            print(f'{KO[r]} {pn}: 장면 {res[key]["n_img"]}  NDVI base {s["base_NDVI"]:.4f}  ({time.time()-t0:.0f}초)')
        except Exception as ex:
            print(f'{KO[r]} {pn}: 실패 — {str(ex)[:200]}')
done = sum(f'{r}|{p}' in res for r in TRAIN_REGIONS for p in ('new', 'old'))
print(f'\n{done}/16 완료' + ('' if done == 16 else '  → 이 셀을 다시 실행'))

인제 new: 장면 83  NDVI base 0.8789  (10초)
인제 old: 장면 58  NDVI base 0.8699  (4초)
홍천 new: 장면 127  NDVI base 0.8749  (17초)
홍천 old: 장면 88  NDVI base 0.8740  (9초)
강릉 new: 장면 180  NDVI base 0.8726  (10초)
강릉 old: 장면 119  NDVI base 0.8799  (7초)
평창 new: 장면 41  NDVI base 0.8762  (7초)
평창 old: 장면 24  NDVI base 0.8848  (4초)
봉화 new: 장면 137  NDVI base 0.8750  (14초)
봉화 old: 장면 86  NDVI base 0.8542  (11초)
금산 new: 장면 49  NDVI base 0.8784  (11초)
금산 old: 장면 35  NDVI base 0.8645  (5초)
거창 new: 장면 100  NDVI base 0.8742  (15초)
거창 old: 장면 74  NDVI base 0.8569  (9초)
순천 new: 장면 100  NDVI base 0.8651  (11초)
순천 old: 장면 66  NDVI base 0.8522  (6초)

16/16 완료


In [7]:
# [5] 표 + 사전 기록 규칙대로 판정
assert len(res) == 16, '[4]를 다시 실행해 16개를 채우세요'
zA = pd.read_csv(f'{OUT}/exp18_A_z_by_feature.csv', index_col=0)       # 18 캐시 이동 (열 = 한글 지역명)

rows = []
for r in TRAIN_REGIONS:
    n, o = res[f'{r}|new']['stats'], res[f'{r}|old']['stats']
    row = {'region': KO[r]}
    for b in ['NDVI', 'B8', 'B2', 'B11', 'AOT']:
        row[f'gap_{b}_base']  = o[f'base_{b}']  - n[f'base_{b}']
        row[f'gap_{b}_csp']   = o[f'csp_{b}']   - n[f'csp_{b}']
        row[f'gap_{b}_month'] = o[f'month_{b}'] - n[f'month_{b}']
        row[f'gap_{b}_noS2C'] = o[f'base_{b}']  - n[f'noS2C_{b}']
    row['z18_s_ndvi'] = zA.loc['s_ndvi', KO[r]]
    for pn, s in (('신', n), ('구', o)):
        row[f'SCL7%_{pn}'] = 100 * s['rec_scl7share']
        row[f'csp남음%_{pn}'] = 100 * s['rec_csp_keep']
        for m in SUMMER:
            row[f'관측{m}월_{pn}'] = s[f'rec_nvalid_m{m}']
    row['장면_신'], row['장면_구'] = res[f'{r}|new']['n_img'], res[f'{r}|old']['n_img']
    rows.append(row)
T = pd.DataFrame(rows).set_index('region')
T.to_csv(f'{OUT}/exp19_summer.csv')

pd.set_option('display.width', 250)
show = ['gap_NDVI_base', 'gap_NDVI_csp', 'gap_NDVI_noS2C', 'gap_NDVI_month', 'z18_s_ndvi']
print('여름 숲 NDVI 차이 (2022–23 − 2024–25). 0에 가까워질수록 그 용의자를 바꿨을 때 차이가 사라진 것\n')
print(T[show].round(4).to_string())
print('\n[기록] 근적외선 B8 차이\n' + T[[c for c in T.columns if c.startswith('gap_B8_')]].round(4).to_string())
print('\n[기록] 파랑 B2 · AOT 차이\n' + T[['gap_B2_base', 'gap_B2_csp', 'gap_AOT_base']].round(4).to_string())
print('\n[기록] 관측\n' + T[[c for c in T.columns if c.startswith(('SCL7', 'csp남음', '관측', '장면'))]].round(1).to_string())

# 측정 믿을 수 있나
agree = int((np.sign(T['gap_NDVI_base']) == np.sign(T['z18_s_ndvi'])).sum())
print(f'\n검산: base 차이 부호 = 18 캐시 s_ndvi 이동 부호  {agree}/8  (7 이상이어야 판정)')

verdict = {}
if agree >= 7:
    big = T.loc[[KO[r] for r in BIG]]
    for X in ('csp', 'noS2C', 'month'):
        ratio = (big[f'gap_NDVI_{X}'].abs() / big['gap_NDVI_base'].abs())
        n_ok = int((ratio <= 0.5).sum())
        verdict[X] = dict(n_ok=n_ok, support=n_ok >= 4, mean_shrink=float(1 - ratio.mean()))
        print(f'  {X:6s}: 반 이상 줄어든 곳 {n_ok}/5  (5곳 평균 줄어든 비율 {1 - ratio.mean():+.0%})'
              f'  → {"지지" if n_ok >= 4 else "지지 안 됨"}')
    sup = sorted([x for x in verdict if verdict[x]['support']], key=lambda x: -verdict[x]['mean_shrink'])
    pr = json.load(open(PR_PATH, encoding='utf-8'))
    print('\n' + '=' * 60)
    print('판정:', ', '.join(sup) if sup else '없음')
    print('사전 기록의 다음 단계:', pr['next_step'][sup[0] if sup else '없음'])
    print('=' * 60)
else:
    print('→ 측정이 18 캐시와 안 맞음. 판정하지 않음 — 출력 그대로 보고')
json.dump(dict(agree=agree, verdict=verdict), open(f'{OUT}/exp19_judgement.json', 'w'), indent=1)

여름 숲 NDVI 차이 (2022–23 − 2024–25). 0에 가까워질수록 그 용의자를 바꿨을 때 차이가 사라진 것

        gap_NDVI_base  gap_NDVI_csp  gap_NDVI_noS2C  gap_NDVI_month  z18_s_ndvi
region                                                                         
인제            -0.0090       -0.0096         -0.0050         -0.0075     -0.2443
홍천            -0.0009       -0.0013          0.0032         -0.0067     -0.0951
강릉             0.0072        0.0076          0.0111          0.0059      0.0958
평창             0.0086        0.0082          0.0131          0.0070      0.2467
봉화            -0.0208       -0.0207         -0.0179         -0.0172     -0.5218
금산            -0.0139       -0.0155         -0.0073         -0.0157     -0.2861
거창            -0.0172       -0.0164         -0.0172         -0.0184     -0.4979
순천            -0.0128       -0.0123         -0.0116         -0.0187     -0.3811

[기록] 근적외선 B8 차이
        gap_B8_base  gap_B8_csp  gap_B8_month  gap_B8_noS2C
region                                                 